# W05B B · 검정·예측·웹 앱

2026-09-30 · **R 커널** · 런타임→모두 실행. 연결이 끊기면 준비 셀부터 다시 실행하세요.

![같은 조건을 다른 척도 또는 구간 대상으로 비교한 수업 그림](https://raw.githubusercontent.com/lunalab-ai/regre/2026-fall-w05b/course/notion/w05b/assets/b05-one-condition.png)

수업의 R 계산으로 만든 그림입니다. 출력이 저장되어 있어도 현재 객체가 살아 있다는 뜻은 아닙니다.


**R 실습**입니다. 온라인 R 또는 R Colab A/B 중 한 환경을 선택하세요. 먼저 예상하고, 실행한 뒤 단위와 비교 조건을 포함해 설명합니다. 학생 연습 셀은 주석으로 되어 있어 기본 흐름을 막지 않습니다. `#`를 지우고 필요한 부분을 완성하세요. 데이터는 R 내장 attitude로 자동 준비되며 파일 업로드·API 키·패키지 설치가 필요 없습니다. 다운로드 R/QMD는 data 폴더를 함께 보관하세요.

공통 소스는 고정 버전 **2026-fall-w05b**입니다. `source()`는 정의를 읽고, `lm()`은 모형을 적합하며, 표와 구간 함수는 그 적합 결과를 읽습니다. [API 전체 안내](https://github.com/lunalab-ai/regre/blob/2026-fall-w05b/src/mlr-inference-api.md). 세션을 재시작하면 준비 셀부터 다시 실행하세요.


In [ ]:
# 공개 고정 버전과 동일한 소스만 사용합니다. MD5는 파일 동일성 확인용입니다.
source_url <- "https://raw.githubusercontent.com/lunalab-ai/regre/2026-fall-w05b/src/mlr-inference.R"
expected_md5 <- "220bb19be470b1d35e13f48ed562ab96"
local_candidates <- c("src/mlr-inference.R", "../../src/mlr-inference.R")
existing <- local_candidates[file.exists(local_candidates)]
if (length(existing)) {
  helper_path <- existing[1]; helper_origin <- "동봉 소스"
} else {
  cache_dir <- file.path(path.expand("~"),".cache","regre","2026-fall-w05b")
  dir.create(cache_dir,recursive=TRUE,showWarnings=FALSE)
  helper_path <- file.path(cache_dir,"mlr-inference.R")
  helper_origin <- if(file.exists(helper_path)) "검증할 캐시" else "고정 버전 최초 다운로드"
  options(timeout=120)
  if (!file.exists(helper_path)) {
    download_status <- download.file(source_url,helper_path,mode="wb",quiet=FALSE)
    if(download_status!=0) stop("공통 소스 다운로드 실패: 인터넷을 확인하고 준비 셀을 다시 실행하세요.")
  }
}
if (unname(tools::md5sum(helper_path))!=expected_md5)
  stop("소스 해시 불일치: 변경한 소스를 별도 보관한 뒤 고정 버전 파일을 다시 받으세요.")
source(helper_path)
cat(helper_origin,"|",R.version.string,"\n")
d <- datasets::attitude
fit2 <- lm(rating~complaints+learning,data=d)
fit6 <- lm(rating~.,data=d)
stopifnot(nrow(d)==30,df.residual(fit2)==27)

## 6. 계수표와 부분 F검정

[mlr_coefficient_table 정의](https://github.com/lunalab-ai/regre/blob/2026-fall-w05b/src/mlr-inference.R#L21)는 적합된 lm을 받아 계수 순서의 estimate/SE/t/p/CI 표를 만듭니다. level=.95, null=0이 기본이며 재적합하지 않습니다. [mlr_nested_f 정의](https://github.com/lunalab-ai/regre/blob/2026-fall-w05b/src/mlr-inference.R#L37)는 두 lm의 반응·행 순서·내포성을 확인하고 q, df_full, SSE, F, p 한 행을 반환합니다.


In [ ]:
coefficient_table <- mlr_coefficient_table(fit6)
coefficient_table
partial_F <- mlr_nested_f(fit2,fit6)
partial_F
anova(fit2,fit6)

### L5 · 예측–실행–설명

부분 F의 분자와 분모를 직접 만들고 q=4, 분모 자유도=23을 확인하세요. 이어 두 변수 모형에서 learning의 계수가 0이라는 한 제약에 대해 F=t²를 확인하세요.


In [ ]:
# manual_F <- ((deviance(fit2)-deviance(fit6))/...)/(deviance(fit6)/...)
# single_F <- mlr_nested_f(lm(rating~complaints,data=d),fit2)
# t_value <- ...

## 7. 동일성·합 제약: 어느 계수를 묶었는가

[mlr_linear_test 정의](https://github.com/lunalab-ai/regre/blob/2026-fall-w05b/src/mlr-inference.R#L59)는 적합된 lm, q×k 제약행렬 L, 귀무값 r(기본 0)을 받아 F 표를 반환합니다. L의 열은 coef 순서와 같아야 합니다. `vcov()`의 공분산을 포함합니다. 아래 L=(0,1,-1)은 두 기울기 차이=0입니다.


In [ ]:
names(coef(fit2))
equal_test <- mlr_linear_test(fit2,L=matrix(c(0,1,-1),nrow=1))
sum_test <- mlr_linear_test(fit2,L=matrix(c(0,1,1),nrow=1),r=1)
equal_test
sum_test

### L6 · 예측–실행–설명

합 제약을 변환 반응 회귀로 적합한 뒤 원래 rating의 예측과 SSE로 복원하세요. 일반 선형 가설과 F가 같은지 검산하세요. 두 반응의 모형을 anova로 직접 비교하지 마세요.


In [ ]:
# constrained <- lm(I(rating-learning)~I(complaints-learning),data=d)
# restored_y <- ...
# restricted_SSE <- ...

## 8. 같은 조건의 평균 CI와 새 부서 PI

[mlr_interval_comparison 정의](https://github.com/lunalab-ai/regre/blob/2026-fall-w05b/src/mlr-inference.R#L77)는 lm과 원 단위 새 조건 data.frame을 받아 조건마다 confidence/prediction 두 행을 반환합니다. fit/lower/upper/width/h0/level을 읽으세요. level=.95, noise_multiplier=1이 기본이며 배율을 바꾸면 실제 재적합이 아닌 가상 오차 시나리오입니다.


In [ ]:
new_department <- data.frame(complaints=mean(d$complaints),learning=mean(d$learning))
intervals <- mlr_interval_comparison(fit2,new_department)
intervals
plot(intervals$fit,2:1,xlim=range(c(intervals$lower,intervals$upper)),ylim=c(.5,2.5),
     yaxt="n",pch=19,xlab="Predicted rating",ylab="")
axis(2,at=2:1,labels=c("Mean CI","New PI"),las=1)
segments(intervals$lower,2:1,intervals$upper,2:1,lwd=4,col=c("#087f8c","#d95d52"))

### L7 · 예측–실행–설명

level=.99의 두 구간을 계산하고 중심과 폭을 95%와 비교하세요. 다음 부서 하나에 해당하는 행을 선택하세요.


In [ ]:
# intervals99 <- mlr_interval_comparison(fit2,new_department,level=...)
# new_case <- intervals99[intervals99$kind==...,]

## 9. 마지막: 웹 앱에서 내 코드 연결하기

[누적 Shiny 편집기](https://lunalab-ai.github.io/regre/apps/w05b/edit/index.html)를 엽니다. W05B 예측 탭에서 구간 종류를 바꿔 보고, 코드의 `W05B-L8` 표식을 찾으세요. 입력→`selected_interval()`→출력의 연결을 수정합니다. Colab 안에서 별도 Shiny 서버를 띄울 필요는 없습니다. 편집기에서 재실행하고 수정 코드를 다운로드합니다.


In [ ]:
# 웹 앱은 위 링크의 편집기에서 실행합니다. 여기서는 같은 계산을 다시 확인합니다.
chosen <- intervals[intervals$kind=="prediction",]
cat("선택 구간의 대상:",chosen$kind,"\n")

### L8 · 예측–실행–설명

앱의 W05B-L8 출력문을 수정하여 선택한 구간의 폭을 소수 둘째자리까지 표시하세요. 평균/새 관측 선택을 바꿨을 때 숫자도 변하는지 재실행으로 확인하세요.


In [ ]:
# 앱 편집기에서 바꿀 문장:
# sprintf("선택 구간 폭: %.2f %%p", ...)
# 이 셀의 주석은 앱 코드로 자동 전송되지 않습니다.

In [ ]:
stopifnot(nrow(d)==30,partial_F$q==4,partial_F$df_full==23,abs(partial_F$F-.5287028)<1e-6,abs(equal_test$F-3.657224)<1e-5,intervals$width[2]>intervals$width[1],abs(intervals$fit[1]-64.6333333)<1e-6)